In [1]:

from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().parent

FEATURES_PATH = (
    PROJECT_ROOT / "data" / "processed" / "features"
    / "respiratory_cycle_features.csv"
)
TRAIN_MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "train_manifest.csv"
)
TEST_MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "test_manifest.csv"
)

# Load existing project data (read-only)
features_df = pd.read_csv(FEATURES_PATH)
train_manifest = pd.read_csv(TRAIN_MANIFEST_PATH)
test_manifest = pd.read_csv(TEST_MANIFEST_PATH)

# Disease labels in a fixed order
ALL_CLASSES = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]

# Exclude patients represented in the official test manifest
test_patients = set(test_manifest["patient_id"].astype(str))

development_manifest = train_manifest[
    ~train_manifest["patient_id"].astype(str).isin(test_patients)
].copy()

# Match manifest cycles to the acoustic feature table
join_keys = [
    "patient_id",
    "recording_id",
    "chest_location",
    "diagnosis",
]

development_keys = development_manifest[join_keys].drop_duplicates()

development_features = features_df.merge(
    development_keys,
    on=join_keys,
    how="inner",
    validate="many_to_one",
)

# Exclude metadata and sound-event labels from disease predictors
metadata_columns = {
    "patient_id",
    "recording_id",
    "chest_location",
    "acquisition_mode",
    "equipment",
    "cycle_number",
    "start_time",
    "end_time",
    "sound_label",
    "crackles",
    "wheezes",
    "diagnosis",
    "processed_path",
    "disease_label",
    "official_split",
    "audio_path",
    "wav_path",
}

candidate_features = [
    col
    for col in development_features.columns
    if col not in metadata_columns
    and pd.api.types.is_numeric_dtype(development_features[col])
]

X_dev = (
    development_features[candidate_features]
    .apply(pd.to_numeric, errors="coerce")
    .copy()
)
y_dev = development_features["diagnosis"].astype(str).copy()
groups_dev = development_features["patient_id"].astype(str).copy()

# Initial verification
print("Project root:", PROJECT_ROOT)
print("XGBoost installed:", importlib.util.find_spec("xgboost") is not None)
print("Development matrix:", X_dev.shape)
print("Number of classes:", y_dev.nunique())
print("Unique patients:", groups_dev.nunique())
print("Acoustic features:", len(candidate_features))
print("Missing values:", int(X_dev.isna().sum().sum()))
print(
    "Infinite values:",
    int(np.isinf(X_dev.to_numpy(dtype=float)).sum()),
)
print(
    "Patient overlap with official test:",
    len(set(groups_dev) & test_patients),
)

print("\nClass distribution:")
class_summary = pd.DataFrame({
    "diagnosis": y_dev,
    "patient_id": groups_dev,
}).groupby("diagnosis").agg(
    cycles=("diagnosis", "size"),
    patients=("patient_id", "nunique"),
).reindex(ALL_CLASSES)

class_summary["cycle_percentage"] = (
    100 * class_summary["cycles"] / len(y_dev)
).round(2)

display(class_summary)

assert X_dev.shape == (4025, 39)
assert len(y_dev) == len(groups_dev) == 4025
assert set(y_dev.unique()) == set(ALL_CLASSES)
assert X_dev.notna().all().all()
assert np.isfinite(X_dev.to_numpy(dtype=float)).all()
assert not (set(groups_dev) & test_patients)

print("\nPASS: development dataset is ready.")


Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
XGBoost installed: True
Development matrix: (4025, 39)
Number of classes: 8
Unique patients: 77
Acoustic features: 39
Missing values: 0
Infinite values: 0
Patient overlap with official test: 0

Class distribution:


,cycles,patients,cycle_percentage
diagnosis,,,
Asthma,6,1,0.15
Bronchiectasis,90,6,2.24
Bronchiolitis,74,3,1.84
COPD,3220,37,80.00
Healthy,147,12,3.65
LRTI,32,2,0.80
Pneumonia,285,6,7.08
URTI,171,10,4.25



PASS: development dataset is ready.


In [2]:

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

import xgboost as xgb

# Fixed label order and numeric encoding for XGBoost
class_to_id = {label: idx for idx, label in enumerate(ALL_CLASSES)}
id_to_class = {idx: label for label, idx in class_to_id.items()}

y_encoded = y_dev.map(class_to_id).astype(int)

# Use identical patient-grouped folds for fair comparison with SVM
cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

fold_splits = list(cv.split(X_dev, y_encoded, groups=groups_dev))

# Verify the folds and inspect training-fold class weights
for fold_number, (train_idx, val_idx) in enumerate(fold_splits, start=1):
    train_patients = set(groups_dev.iloc[train_idx])
    val_patients = set(groups_dev.iloc[val_idx])

    assert train_patients.isdisjoint(val_patients)

    y_train_fold = y_encoded.iloc[train_idx]
    present_classes = np.sort(y_train_fold.unique())

    weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_train_fold,
    )
    class_weight_map = dict(zip(present_classes, weights))

    print(f"\nFold {fold_number}")
    print("Training cycles:", len(train_idx))
    print("Validation cycles:", len(val_idx))
    print("Training patients:", len(train_patients))
    print("Validation patients:", len(val_patients))
    print("Patient overlap:", len(train_patients & val_patients))
    print("Classes present in training:", len(present_classes))
    print("Class weights:")
    for class_id in present_classes:
        print(
            f"  {id_to_class[int(class_id)]:18s} "
            f"{class_weight_map[class_id]:.3f}"
        )

print("\nPASS: folds are patient-disjoint and weights are fold-specific.")



Fold 1
Training cycles: 2685
Validation cycles: 1340
Training patients: 55
Validation patients: 22
Patient overlap: 0
Classes present in training: 8
Class weights:
  Asthma             55.938
  Bronchiectasis     6.849
  Bronchiolitis      6.102
  COPD               0.156
  Healthy            3.460
  LRTI               10.488
  Pneumonia          1.766
  URTI               3.079

Fold 2
Training cycles: 2679
Validation cycles: 1346
Training patients: 50
Validation patients: 27
Patient overlap: 0
Classes present in training: 8
Class weights:
  Asthma             55.812
  Bronchiectasis     5.074
  Bronchiolitis      7.125
  COPD               0.156
  Healthy            3.383
  LRTI               13.953
  Pneumonia          1.936
  URTI               2.838

Fold 3
Training cycles: 2686
Validation cycles: 1339
Training patients: 49
Validation patients: 28
Patient overlap: 0
Classes present in training: 7
Class weights:
  Bronchiectasis     5.903
  Bronchiolitis      8.342
  COPD         

In [4]:

from sklearn.utils.class_weight import compute_sample_weight

xgb_oof_predictions = np.full(len(y_encoded), -1, dtype=int)
xgb_oof_probabilities = np.zeros(
    (len(y_encoded), len(ALL_CLASSES)),
    dtype=float,
)
xgb_fold_results = []

for fold_number, (train_idx, val_idx) in enumerate(fold_splits, start=1):
    X_train = X_dev.iloc[train_idx]
    X_val = X_dev.iloc[val_idx]

    y_train_global = y_encoded.iloc[train_idx]
    y_val_global = y_encoded.iloc[val_idx]

    train_groups = set(groups_dev.iloc[train_idx])
    val_groups = set(groups_dev.iloc[val_idx])
    assert train_groups.isdisjoint(val_groups)

    # Remap classes present in this training fold to 0..K-1.
    # Keep the mapping back to the original global class IDs.
    present_classes = np.sort(y_train_global.unique())
    global_to_local = {
        int(global_id): local_id
        for local_id, global_id in enumerate(present_classes)
    }
    local_to_global = {
        local_id: global_id
        for global_id, local_id in global_to_local.items()
    }

    y_train_local = y_train_global.map(global_to_local).astype(int)

    sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train_local,
    )

    model = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=len(present_classes),
        eval_metric="mlogloss",
        n_estimators=400,
        learning_rate=0.05,
        max_depth=3,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=5.0,
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(
        X_train,
        y_train_local,
        sample_weight=sample_weights,
        verbose=False,
    )

    local_probabilities = model.predict_proba(X_val)
    local_predictions = np.argmax(local_probabilities, axis=1)

    # Restore the global class IDs; absent training classes get probability 0.
    global_probabilities = np.zeros(
        (len(val_idx), len(ALL_CLASSES)),
        dtype=float,
    )

    for local_id, global_id in local_to_global.items():
        global_probabilities[:, global_id] = local_probabilities[:, local_id]

    global_predictions = np.array([
        local_to_global[int(local_id)]
        for local_id in local_predictions
    ])

    xgb_oof_predictions[val_idx] = global_predictions
    xgb_oof_probabilities[val_idx] = global_probabilities

    fold_metrics = {
        "fold": fold_number,
        "train_patients": len(train_groups),
        "validation_patients": len(val_groups),
        "training_classes": len(present_classes),
        "accuracy": accuracy_score(y_val_global, global_predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y_val_global, global_predictions
        ),
        "macro_f1": f1_score(
            y_val_global,
            global_predictions,
            labels=list(range(len(ALL_CLASSES))),
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_val_global,
            global_predictions,
            average="weighted",
            zero_division=0,
        ),
    }
    xgb_fold_results.append(fold_metrics)

    print(
        f"Fold {fold_number}/3 | "
        f"training classes={len(present_classes)} | "
        f"accuracy={fold_metrics['accuracy']:.4f} | "
        f"balanced accuracy={fold_metrics['balanced_accuracy']:.4f} | "
        f"macro-F1={fold_metrics['macro_f1']:.4f}"
    )

assert np.all(xgb_oof_predictions >= 0)
assert np.isfinite(xgb_oof_probabilities).all()
assert np.allclose(xgb_oof_probabilities.sum(axis=1), 1.0)

xgb_fold_results_df = pd.DataFrame(xgb_fold_results)

print("\nPER-FOLD RESULTS")
display(xgb_fold_results_df.round(4))

print("\nMEAN AND STANDARD DEVIATION ACROSS FOLDS")
display(
    xgb_fold_results_df[
        ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
    ].agg(["mean", "std"]).round(4)
)

print("\nPOOLED OUT-OF-FOLD RESULTS")
print("Accuracy:", round(
    accuracy_score(y_encoded, xgb_oof_predictions), 4
))
print("Balanced accuracy:", round(
    balanced_accuracy_score(y_encoded, xgb_oof_predictions), 4
))
print("Macro-F1:", round(f1_score(
    y_encoded,
    xgb_oof_predictions,
    labels=list(range(len(ALL_CLASSES))),
    average="macro",
    zero_division=0,
), 4))
print("Weighted-F1:", round(f1_score(
    y_encoded,
    xgb_oof_predictions,
    average="weighted",
    zero_division=0,
), 4))

print("\nPOOLED CLASSIFICATION REPORT")
print(classification_report(
    y_encoded,
    xgb_oof_predictions,
    labels=list(range(len(ALL_CLASSES))),
    target_names=ALL_CLASSES,
    zero_division=0,
))

print("POOLED CONFUSION MATRIX (rows=true, columns=predicted)")
display(pd.DataFrame(
    confusion_matrix(
        y_encoded,
        xgb_oof_predictions,
        labels=list(range(len(ALL_CLASSES))),
    ),
    index=ALL_CLASSES,
    columns=ALL_CLASSES,
))


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Fold 1/3 | training classes=8 | accuracy=0.7634 | balanced accuracy=0.3305 | macro-F1=0.2449
Fold 2/3 | training classes=8 | accuracy=0.7897 | balanced accuracy=0.2879 | macro-F1=0.2548
Fold 3/3 | training classes=7 | accuracy=0.7289 | balanced accuracy=0.2152 | macro-F1=0.1955

PER-FOLD RESULTS


,fold,train_patients,validation_patients,training_classes,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,1,55,22,8,0.7634,0.3305,0.2449,0.7711
1,2,50,27,8,0.7897,0.2879,0.2548,0.7740
2,3,49,28,7,0.7289,0.2152,0.1955,0.7376



MEAN AND STANDARD DEVIATION ACROSS FOLDS


,accuracy,balanced_accuracy,macro_f1,weighted_f1
mean,0.7607,0.2779,0.2317,0.7609
std,0.0305,0.0583,0.0317,0.0203



POOLED OUT-OF-FOLD RESULTS
Accuracy: 0.7607
Balanced accuracy: 0.2398
Macro-F1: 0.232
Weighted-F1: 0.7608

POOLED CLASSIFICATION REPORT
                precision    recall  f1-score   support

        Asthma       0.00      0.00      0.00         6
Bronchiectasis       0.17      0.12      0.14        90
 Bronchiolitis       0.00      0.00      0.00        74
          COPD       0.90      0.90      0.90      3220
       Healthy       0.34      0.48      0.40       147
          LRTI       0.00      0.00      0.00        32
     Pneumonia       0.21      0.21      0.21       285
          URTI       0.20      0.21      0.21       171

      accuracy                           0.76      4025
     macro avg       0.23      0.24      0.23      4025
  weighted avg       0.76      0.76      0.76      4025

POOLED CONFUSION MATRIX (rows=true, columns=predicted)


,Asthma,Bronchiectasis,Bronchiolitis,COPD,Healthy,LRTI,Pneumonia,URTI
Asthma,0,0,0,3,3,0,0,0
Bronchiectasis,0,11,0,72,0,0,0,7
Bronchiolitis,0,0,0,20,6,0,15,33
COPD,0,52,41,2885,63,2,159,18
Healthy,0,1,3,21,71,4,21,26
LRTI,0,0,0,0,6,0,6,20
Pneumonia,0,0,11,146,28,5,59,36
URTI,0,1,16,49,34,19,16,36


In [5]:

from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

def evaluate_xgb_weight_strategy(strategy_name):
    oof_pred = np.full(len(y_encoded), -1, dtype=int)
    fold_rows = []

    for fold_number, (train_idx, val_idx) in enumerate(fold_splits, start=1):
        X_train = X_dev.iloc[train_idx]
        X_val = X_dev.iloc[val_idx]
        y_train_global = y_encoded.iloc[train_idx]
        y_val = y_encoded.iloc[val_idx]

        # Fold-local label mapping supports folds missing a class
        present_classes = np.sort(y_train_global.unique())
        global_to_local = {
            int(global_id): local_id
            for local_id, global_id in enumerate(present_classes)
        }
        local_to_global = {
            local_id: global_id
            for global_id, local_id in global_to_local.items()
        }
        y_train_local = y_train_global.map(global_to_local).astype(int)

        if strategy_name == "unweighted":
            sample_weights = None

        elif strategy_name == "balanced":
            sample_weights = compute_sample_weight(
                "balanced", y_train_local
            )

        elif strategy_name == "sqrt_balanced":
            # Square-root inverse-frequency weights, normalized to mean 1
            counts = y_train_local.value_counts().sort_index()
            class_weights = np.sqrt(
                len(y_train_local) / (len(counts) * counts)
            )
            class_weight_map = class_weights.to_dict()
            sample_weights = y_train_local.map(class_weight_map).to_numpy()
            sample_weights = sample_weights / sample_weights.mean()

        else:
            raise ValueError(f"Unknown strategy: {strategy_name}")

        model = xgb.XGBClassifier(
            objective="multi:softprob",
            num_class=len(present_classes),
            eval_metric="mlogloss",
            n_estimators=400,
            learning_rate=0.05,
            max_depth=3,
            min_child_weight=5,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=5.0,
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
        )

        fit_args = {}
        if sample_weights is not None:
            fit_args["sample_weight"] = sample_weights

        model.fit(X_train, y_train_local, verbose=False, **fit_args)
        local_pred = model.predict(X_val).astype(int)
        global_pred = np.array([
            local_to_global[int(local_id)] for local_id in local_pred
        ])
        oof_pred[val_idx] = global_pred

        fold_rows.append({
            "fold": fold_number,
            "accuracy": accuracy_score(y_val, global_pred),
            "balanced_accuracy": balanced_accuracy_score(y_val, global_pred),
            "macro_f1": f1_score(
                y_val, global_pred,
                labels=list(range(len(ALL_CLASSES))),
                average="macro",
                zero_division=0,
            ),
        })

    assert np.all(oof_pred >= 0)

    summary = {
        "strategy": strategy_name,
        "pooled_accuracy": accuracy_score(y_encoded, oof_pred),
        "pooled_balanced_accuracy": balanced_accuracy_score(
            y_encoded, oof_pred
        ),
        "pooled_macro_f1": f1_score(
            y_encoded, oof_pred,
            labels=list(range(len(ALL_CLASSES))),
            average="macro",
            zero_division=0,
        ),
        "pooled_weighted_f1": f1_score(
            y_encoded, oof_pred,
            average="weighted",
            zero_division=0,
        ),
    }

    class_recalls = {}
    for class_id, class_name in enumerate(ALL_CLASSES):
        mask = y_encoded.to_numpy() == class_id
        class_recalls[f"recall_{class_name}"] = (
            float(np.mean(oof_pred[mask] == class_id))
            if mask.any() else np.nan
        )

    return summary, oof_pred, pd.DataFrame(fold_rows), class_recalls


strategy_results = []
strategy_predictions = {}
strategy_fold_results = {}
strategy_class_recalls = {}

for strategy in ["unweighted", "sqrt_balanced"]:
    summary, predictions, fold_df, recalls = evaluate_xgb_weight_strategy(
        strategy
    )
    strategy_results.append(summary)
    strategy_predictions[strategy] = predictions
    strategy_fold_results[strategy] = fold_df
    strategy_class_recalls[strategy] = recalls

results_table = pd.DataFrame(strategy_results)
display(results_table.round(4))

recall_table = pd.DataFrame(strategy_class_recalls).T
display(recall_table.round(4))

print("Baseline balanced-weighted pooled results:")
print("Accuracy:", round(accuracy_score(y_encoded, xgb_oof_predictions), 4))
print("Balanced accuracy:", round(
    balanced_accuracy_score(y_encoded, xgb_oof_predictions), 4
))
print("Macro-F1:", round(f1_score(
    y_encoded, xgb_oof_predictions,
    labels=list(range(len(ALL_CLASSES))),
    average="macro",
    zero_division=0,
), 4))


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,strategy,pooled_accuracy,pooled_balanced_accuracy,pooled_macro_f1,pooled_weighted_f1
0,unweighted,0.7908,0.1798,0.1858,0.7482
1,sqrt_balanced,0.7771,0.1957,0.1985,0.7504


,recall_Asthma,recall_Bronchiectasis,recall_Bronchiolitis,recall_COPD,recall_Healthy,recall_LRTI,recall_Pneumonia,recall_URTI
unweighted,0.0,0.0000,0.0,0.9634,0.2585,0.0,0.0526,0.1637
sqrt_balanced,0.0,0.0222,0.0,0.9385,0.3605,0.0,0.0807,0.1637


Baseline balanced-weighted pooled results:
Accuracy: 0.7607
Balanced accuracy: 0.2398
Macro-F1: 0.232


In [6]:

from itertools import product
from sklearn.utils.class_weight import compute_sample_weight

# Small, controlled search to avoid an unnecessarily large experiment.
xgb_configs = [
    {"max_depth": 2, "learning_rate": 0.05, "min_child_weight": 5,
     "reg_lambda": 5.0, "reg_alpha": 0.1},
    {"max_depth": 3, "learning_rate": 0.05, "min_child_weight": 5,
     "reg_lambda": 5.0, "reg_alpha": 0.1},
    {"max_depth": 4, "learning_rate": 0.05, "min_child_weight": 5,
     "reg_lambda": 5.0, "reg_alpha": 0.1},
    {"max_depth": 3, "learning_rate": 0.03, "min_child_weight": 5,
     "reg_lambda": 10.0, "reg_alpha": 0.5},
    {"max_depth": 3, "learning_rate": 0.08, "min_child_weight": 10,
     "reg_lambda": 10.0, "reg_alpha": 0.5},
    {"max_depth": 2, "learning_rate": 0.08, "min_child_weight": 10,
     "reg_lambda": 10.0, "reg_alpha": 0.5},
]

tuning_rows = []
tuning_oof_predictions = {}

for config_id, config in enumerate(xgb_configs, start=1):
    oof_pred = np.full(len(y_encoded), -1, dtype=int)

    for fold_number, (train_idx, val_idx) in enumerate(fold_splits, start=1):
        X_train = X_dev.iloc[train_idx]
        X_val = X_dev.iloc[val_idx]
        y_train_global = y_encoded.iloc[train_idx]
        y_val = y_encoded.iloc[val_idx]

        present_classes = np.sort(y_train_global.unique())
        global_to_local = {
            int(global_id): local_id
            for local_id, global_id in enumerate(present_classes)
        }
        local_to_global = {
            local_id: global_id
            for global_id, local_id in global_to_local.items()
        }
        y_train_local = y_train_global.map(global_to_local).astype(int)

        sample_weights = compute_sample_weight(
            class_weight="balanced",
            y=y_train_local,
        )

        model = xgb.XGBClassifier(
            objective="multi:softprob",
            num_class=len(present_classes),
            eval_metric="mlogloss",
            n_estimators=400,
            learning_rate=config["learning_rate"],
            max_depth=config["max_depth"],
            min_child_weight=config["min_child_weight"],
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=config["reg_alpha"],
            reg_lambda=config["reg_lambda"],
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
        )

        model.fit(
            X_train,
            y_train_local,
            sample_weight=sample_weights,
            verbose=False,
        )

        local_pred = model.predict(X_val).astype(int)
        oof_pred[val_idx] = np.array([
            local_to_global[int(local_id)]
            for local_id in local_pred
        ])

    assert np.all(oof_pred >= 0)
    tuning_oof_predictions[config_id] = oof_pred

    tuning_rows.append({
        "config_id": config_id,
        **config,
        "pooled_accuracy": accuracy_score(y_encoded, oof_pred),
        "pooled_balanced_accuracy": balanced_accuracy_score(
            y_encoded, oof_pred
        ),
        "pooled_macro_f1": f1_score(
            y_encoded,
            oof_pred,
            labels=list(range(len(ALL_CLASSES))),
            average="macro",
            zero_division=0,
        ),
        "pooled_weighted_f1": f1_score(
            y_encoded, oof_pred,
            average="weighted",
            zero_division=0,
        ),
    })

    print(f"Completed configuration {config_id}/{len(xgb_configs)}")

tuning_results = pd.DataFrame(tuning_rows).sort_values(
    "pooled_macro_f1",
    ascending=False,
).reset_index(drop=True)

print("\nXGBOOST CONFIGURATION COMPARISON")
display(tuning_results.round(4))


Completed configuration 1/6
Completed configuration 2/6
Completed configuration 3/6
Completed configuration 4/6
Completed configuration 5/6
Completed configuration 6/6

XGBOOST CONFIGURATION COMPARISON


,config_id,max_depth,learning_rate,min_child_weight,reg_lambda,reg_alpha,pooled_accuracy,pooled_balanced_accuracy,pooled_macro_f1,pooled_weighted_f1
0,6,2,0.08,10,10.0,0.5,0.7453,0.2715,0.2519,0.7606
1,4,3,0.03,5,10.0,0.5,0.7478,0.2756,0.2518,0.7644
2,1,2,0.05,5,5.0,0.1,0.7434,0.2712,0.2473,0.7619
3,5,3,0.08,10,10.0,0.5,0.7573,0.2506,0.2404,0.7623
4,3,4,0.05,5,5.0,0.1,0.7660,0.2374,0.2323,0.7617
5,2,3,0.05,5,5.0,0.1,0.7607,0.2398,0.2320,0.7608


In [7]:

from sklearn.metrics import classification_report, confusion_matrix

candidate_ids = [6, 4]
candidate_reports = []

for config_id in candidate_ids:
    predictions = tuning_oof_predictions[config_id]

    print(f"\n{'=' * 18} CONFIGURATION {config_id} {'=' * 18}")

    print("Accuracy:", round(accuracy_score(y_encoded, predictions), 4))
    print("Balanced accuracy:", round(
        balanced_accuracy_score(y_encoded, predictions), 4
    ))
    print("Macro-F1:", round(f1_score(
        y_encoded,
        predictions,
        labels=list(range(len(ALL_CLASSES))),
        average="macro",
        zero_division=0,
    ), 4))

    report = classification_report(
        y_encoded,
        predictions,
        labels=list(range(len(ALL_CLASSES))),
        target_names=ALL_CLASSES,
        output_dict=True,
        zero_division=0,
    )

    class_df = pd.DataFrame(report).T.loc[ALL_CLASSES, [
        "precision", "recall", "f1-score", "support"
    ]]

    display(class_df.round(4))

    print("Confusion matrix (rows=true, columns=predicted):")
    display(pd.DataFrame(
        confusion_matrix(
            y_encoded,
            predictions,
            labels=list(range(len(ALL_CLASSES))),
        ),
        index=ALL_CLASSES,
        columns=ALL_CLASSES,
    ))

    candidate_reports.append({
        "config_id": config_id,
        "predictions": predictions.copy(),
        "class_report": class_df.copy(),
    })




================== CONFIGURATION 6 ==================
Accuracy: 0.7453
Balanced accuracy: 0.2715
Macro-F1: 0.2519


,precision,recall,f1-score,support
Asthma,0.0000,0.0000,0.0000,6.0
Bronchiectasis,0.2574,0.2889,0.2723,90.0
Bronchiolitis,0.0000,0.0000,0.0000,74.0
COPD,0.9229,0.8627,0.8918,3220.0
Healthy,0.3149,0.5034,0.3874,147.0
LRTI,0.0000,0.0000,0.0000,32.0
Pneumonia,0.2270,0.2947,0.2565,285.0
URTI,0.1939,0.2222,0.2071,171.0


Confusion matrix (rows=true, columns=predicted):


,Asthma,Bronchiectasis,Bronchiolitis,COPD,Healthy,LRTI,Pneumonia,URTI
Asthma,0,0,0,3,3,0,0,0
Bronchiectasis,0,26,0,50,1,0,0,13
Bronchiolitis,0,0,0,15,8,0,15,36
COPD,0,73,43,2778,78,5,220,23
Healthy,0,1,4,7,74,6,24,31
LRTI,0,0,0,0,7,0,8,17
Pneumonia,0,0,12,117,28,6,84,38
URTI,0,1,17,40,36,20,19,38



================== CONFIGURATION 4 ==================
Accuracy: 0.7478
Balanced accuracy: 0.2756
Macro-F1: 0.2518


,precision,recall,f1-score,support
Asthma,0.0000,0.0000,0.0000,6.0
Bronchiectasis,0.2323,0.2556,0.2434,90.0
Bronchiolitis,0.0000,0.0000,0.0000,74.0
COPD,0.9296,0.8618,0.8944,3220.0
Healthy,0.2966,0.5306,0.3805,147.0
LRTI,0.0000,0.0000,0.0000,32.0
Pneumonia,0.2622,0.3404,0.2962,285.0
URTI,0.1859,0.2164,0.2000,171.0


Confusion matrix (rows=true, columns=predicted):


,Asthma,Bronchiectasis,Bronchiolitis,COPD,Healthy,LRTI,Pneumonia,URTI
Asthma,0,0,0,0,6,0,0,0
Bronchiectasis,0,23,0,52,1,0,0,14
Bronchiolitis,0,0,0,14,9,0,15,36
COPD,0,73,50,2775,92,1,208,21
Healthy,0,2,3,11,78,4,20,29
LRTI,0,0,0,0,6,0,5,21
Pneumonia,0,0,12,92,36,7,97,41
URTI,0,1,14,41,35,18,25,37


In [8]:

# Diagnostic inspection of Configuration 4 out-of-fold predictions

best_config_id = 4
best_oof_pred = tuning_oof_predictions[best_config_id]

print("True class counts:")
display(
    y_dev.value_counts()
    .reindex(ALL_CLASSES, fill_value=0)
    .rename("true_cycles")
    .to_frame()
)

predicted_labels = pd.Series(best_oof_pred).map(id_to_class)

print("\nPredicted class counts:")
display(
    predicted_labels.value_counts()
    .reindex(ALL_CLASSES, fill_value=0)
    .rename("predicted_cycles")
    .to_frame()
)

print("\nPer-class recall:")
recall_by_class = {}
for class_id, class_name in enumerate(ALL_CLASSES):
    mask = y_encoded.to_numpy() == class_id
    recall_by_class[class_name] = float(
        np.mean(best_oof_pred[mask] == class_id)
    )

display(
    pd.Series(recall_by_class, name="recall")
    .to_frame()
    .round(4)
)

print("\nCurrent Configuration 4 parameters:")
display(
    tuning_results.loc[
        tuning_results["config_id"] == 4
    ].reset_index(drop=True)
)


True class counts:


,true_cycles
diagnosis,
Asthma,6
Bronchiectasis,90
Bronchiolitis,74
COPD,3220
Healthy,147
LRTI,32
Pneumonia,285
URTI,171



Predicted class counts:


,predicted_cycles
Asthma,0
Bronchiectasis,99
Bronchiolitis,79
COPD,2985
Healthy,263
LRTI,30
Pneumonia,370
URTI,199



Per-class recall:


,recall
Asthma,0.0000
Bronchiectasis,0.2556
Bronchiolitis,0.0000
COPD,0.8618
Healthy,0.5306
LRTI,0.0000
Pneumonia,0.3404
URTI,0.2164



Current Configuration 4 parameters:


,config_id,max_depth,learning_rate,min_child_weight,reg_lambda,reg_alpha,pooled_accuracy,pooled_balanced_accuracy,pooled_macro_f1,pooled_weighted_f1
0,4,3,0.03,5,10.0,0.5,0.747826,0.275587,0.251812,0.764359


In [9]:

from sklearn.utils.class_weight import compute_sample_weight

train_idx, val_idx = fold_splits[0]

y_train_fold = y_encoded.iloc[train_idx].reset_index(drop=True)
patient_train_fold = groups_dev.iloc[train_idx].reset_index(drop=True)

# A. Existing cycle-balanced weights
cycle_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_fold,
)

# B. Patient-balanced weights:
# First give each patient equal total contribution,
# then adjust for disease-class imbalance.
patient_cycle_counts = patient_train_fold.value_counts()
patient_weights = patient_train_fold.map(
    lambda patient: 1.0 / patient_cycle_counts[patient]
).to_numpy()

class_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_fold,
)

patient_balanced_weights = patient_weights * class_weights

# Normalize both strategies to mean 1 for easier comparison
cycle_weights = cycle_weights / cycle_weights.mean()
patient_balanced_weights = (
    patient_balanced_weights / patient_balanced_weights.mean()
)

print("Fold 1 training cycles:", len(train_idx))
print("Fold 1 training patients:", patient_train_fold.nunique())

print("\nCycle-balanced weights:")
print(pd.Series(cycle_weights).describe().round(4))

print("\nPatient-balanced weights:")
print(pd.Series(patient_balanced_weights).describe().round(4))

print("\nMean weight by disease class:")
weight_comparison = pd.DataFrame({
    "diagnosis": y_train_fold.map(id_to_class),
    "cycle_balanced": cycle_weights,
    "patient_balanced": patient_balanced_weights,
}).groupby("diagnosis").agg(
    cycle_balanced_mean=("cycle_balanced", "mean"),
    patient_balanced_mean=("patient_balanced", "mean"),
).reindex(ALL_CLASSES)

display(weight_comparison.round(4))


Fold 1 training cycles: 2685
Fold 1 training patients: 55

Cycle-balanced weights:
count    2685.0000
mean        1.0000
std         3.1597
min         0.1563
25%         0.1563
50%         0.1563
75%         0.1563
max        55.9375
dtype: float64

Patient-balanced weights:
count    2685.0000
mean        1.0000
std         6.7380
min         0.0098
25%         0.0123
50%         0.0180
75%         0.0649
max       135.5557
dtype: float64

Mean weight by disease class:


,cycle_balanced_mean,patient_balanced_mean
diagnosis,,
Asthma,55.9375,135.5557
Bronchiectasis,6.8495,10.1624
Bronchiolitis,6.1023,3.2265
COPD,0.1563,0.0275
Healthy,3.4601,3.6306
LRTI,10.4883,9.5313
Pneumonia,1.7664,0.4055
URTI,3.0791,3.6967


In [10]:

from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

def evaluate_weighting_strategy(strategy_name):
    oof_pred = np.full(len(y_encoded), -1, dtype=int)
    fold_rows = []

    for fold_number, (train_idx, val_idx) in enumerate(fold_splits, start=1):
        X_train = X_dev.iloc[train_idx]
        X_val = X_dev.iloc[val_idx]

        y_train_global = y_encoded.iloc[train_idx]
        y_val = y_encoded.iloc[val_idx]
        patient_train = groups_dev.iloc[train_idx].reset_index(drop=True)

        # Remap classes present in this training fold
        present_classes = np.sort(y_train_global.unique())
        global_to_local = {
            int(g): i for i, g in enumerate(present_classes)
        }
        local_to_global = {
            i: g for g, i in global_to_local.items()
        }
        y_train_local = (
            y_train_global.map(global_to_local).astype(int).reset_index(drop=True)
        )

        # Class weighting, calculated from training data only
        class_weights = compute_sample_weight(
            "balanced",
            y_train_local,
        )

        if strategy_name == "cycle_balanced":
            weights = class_weights

        elif strategy_name == "patient_balanced":
            # Each patient gets equal total weight before class adjustment.
            patient_counts = patient_train.value_counts()
            patient_weights = patient_train.map(
                lambda p: 1.0 / patient_counts[p]
            ).to_numpy()

            weights = patient_weights * class_weights

        else:
            raise ValueError(strategy_name)

        weights = weights / weights.mean()

        model = xgb.XGBClassifier(
            objective="multi:softprob",
            num_class=len(present_classes),
            eval_metric="mlogloss",
            n_estimators=400,
            learning_rate=0.03,
            max_depth=3,
            min_child_weight=5,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.5,
            reg_lambda=10.0,
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
        )

        model.fit(
            X_train,
            y_train_local,
            sample_weight=weights,
            verbose=False,
        )

        local_pred = model.predict(X_val).astype(int)
        global_pred = np.array([
            local_to_global[int(i)] for i in local_pred
        ])
        oof_pred[val_idx] = global_pred

        fold_rows.append({
            "fold": fold_number,
            "accuracy": accuracy_score(y_val, global_pred),
            "balanced_accuracy": balanced_accuracy_score(y_val, global_pred),
            "macro_f1": f1_score(
                y_val,
                global_pred,
                labels=list(range(len(ALL_CLASSES))),
                average="macro",
                zero_division=0,
            ),
            "weighted_f1": f1_score(
                y_val,
                global_pred,
                average="weighted",
                zero_division=0,
            ),
        })

    assert np.all(oof_pred >= 0)

    summary = {
        "strategy": strategy_name,
        "pooled_accuracy": accuracy_score(y_encoded, oof_pred),
        "pooled_balanced_accuracy": balanced_accuracy_score(
            y_encoded, oof_pred
        ),
        "pooled_macro_f1": f1_score(
            y_encoded,
            oof_pred,
            labels=list(range(len(ALL_CLASSES))),
            average="macro",
            zero_division=0,
        ),
        "pooled_weighted_f1": f1_score(
            y_encoded,
            oof_pred,
            average="weighted",
            zero_division=0,
        ),
    }

    recalls = {}
    for class_id, class_name in enumerate(ALL_CLASSES):
        mask = y_encoded.to_numpy() == class_id
        recalls[f"recall_{class_name}"] = float(
            np.mean(oof_pred[mask] == class_id)
        )

    return summary, oof_pred, pd.DataFrame(fold_rows), recalls


weighting_summaries = []
weighting_predictions = {}
weighting_fold_results = {}
weighting_recalls = {}

for strategy in ["cycle_balanced", "patient_balanced"]:
    summary, pred, fold_df, recalls = evaluate_weighting_strategy(strategy)
    weighting_summaries.append(summary)
    weighting_predictions[strategy] = pred
    weighting_fold_results[strategy] = fold_df
    weighting_recalls[strategy] = recalls

print("POOLED OUT-OF-FOLD COMPARISON")
display(pd.DataFrame(weighting_summaries).round(4))

print("\nPER-CLASS RECALL COMPARISON")
display(pd.DataFrame(weighting_recalls).T.round(4))

print("\nFOLD-LEVEL RESULTS")
for strategy, fold_df in weighting_fold_results.items():
    print(f"\n{strategy}")
    display(fold_df.round(4))


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


POOLED OUT-OF-FOLD COMPARISON


,strategy,pooled_accuracy,pooled_balanced_accuracy,pooled_macro_f1,pooled_weighted_f1
0,cycle_balanced,0.7478,0.2756,0.2518,0.7644
1,patient_balanced,0.6176,0.3244,0.2290,0.6814



PER-CLASS RECALL COMPARISON


,recall_Asthma,recall_Bronchiectasis,recall_Bronchiolitis,recall_COPD,recall_Healthy,recall_LRTI,recall_Pneumonia,recall_URTI
cycle_balanced,0.0,0.2556,0.0,0.8618,0.5306,0.0,0.3404,0.2164
patient_balanced,0.0,0.6889,0.0,0.6823,0.6122,0.0,0.2842,0.3275



FOLD-LEVEL RESULTS

cycle_balanced


,fold,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,1,0.7522,0.3667,0.2625,0.7765
1,2,0.7823,0.3543,0.2927,0.7848
2,3,0.7087,0.2382,0.2051,0.7322



patient_balanced


,fold,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,1,0.5903,0.5038,0.2503,0.6617
1,2,0.6738,0.3341,0.2292,0.7185
2,3,0.5885,0.2641,0.1867,0.6593


In [11]:

print("Development feature columns:")
print(development_features.columns.tolist())

print("\nUnique patients:", development_features["patient_id"].nunique())
print("Unique recordings:", development_features["recording_id"].nunique())

print("\nRows per recording:")
display(
    development_features.groupby(
        ["patient_id", "recording_id", "diagnosis"]
    ).size().describe().to_frame("cycles_per_recording").round(2)
)

print("\nDiagnoses per patient:")
patient_diagnoses = (
    development_features.groupby("patient_id")["diagnosis"]
    .nunique()
)
print(patient_diagnoses.value_counts().sort_index())

print("\nRecording identity and diagnosis consistency:")
recording_diagnoses = (
    development_features.groupby(["patient_id", "recording_id"])["diagnosis"]
    .nunique()
)
print("Recordings with multiple diagnosis labels:",
      int((recording_diagnoses > 1).sum()))


Development feature columns:
['patient_id', 'recording_id', 'chest_location', 'acquisition_mode', 'equipment', 'cycle_number', 'start_time', 'end_time', 'sound_label', 'crackles', 'wheezes', 'diagnosis', 'processed_path', 'mean', 'std', 'variance', 'rms', 'max', 'min', 'peak_to_peak', 'median', 'skewness', 'kurtosis', 'energy', 'zero_crossing_rate', 'spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff', 'spectral_flatness', 'spectral_energy', 'dominant_frequency', 'spectral_entropy', 'band_energy_0_500', 'band_energy_500_1000', 'band_energy_1000_1500', 'band_energy_1500_2000', 'mfcc_1_mean', 'mfcc_2_mean', 'mfcc_3_mean', 'mfcc_4_mean', 'mfcc_5_mean', 'mfcc_6_mean', 'mfcc_7_mean', 'mfcc_8_mean', 'mfcc_9_mean', 'mfcc_10_mean', 'mfcc_11_mean', 'mfcc_12_mean', 'mfcc_13_mean', 'wavelet_energy', 'wavelet_entropy', 'wavelet_std']

Unique patients: 77
Unique recordings: 28

Rows per recording:


,cycles_per_recording
count,154.00
mean,26.14
std,21.12
min,4.00
25%,9.00
50%,20.00
75%,36.00
max,123.00



Diagnoses per patient:
diagnosis
1    77
Name: count, dtype: int64

Recording identity and diagnosis consistency:
Recordings with multiple diagnosis labels: 0


In [12]:
# Inspect patient-level support for every diagnosis

patient_class_support = (
    development_features
    .groupby(["diagnosis", "patient_id"])
    .agg(
        cycles=("cycle_number", "count"),
        recordings=("recording_id", "nunique"),
        chest_locations=("chest_location", "nunique")
    )
    .reset_index()
    .sort_values(["diagnosis", "cycles"], ascending=[True, False])
)

for diagnosis in ALL_CLASSES:
    subset = patient_class_support[
        patient_class_support["diagnosis"] == diagnosis
    ]

    print(f"\n{'=' * 65}")
    print(f"{diagnosis}: {len(subset)} patients, "
          f"{subset['cycles'].sum()} cycles")

    if len(subset) > 0:
        display(subset.reset_index(drop=True))

print("\nOverall patient-level class support:")
display(
    patient_class_support.groupby("diagnosis").agg(
        patients=("patient_id", "nunique"),
        total_cycles=("cycles", "sum"),
        min_cycles_per_patient=("cycles", "min"),
        median_cycles_per_patient=("cycles", "median"),
        max_cycles_per_patient=("cycles", "max")
    ).reindex(ALL_CLASSES)
)


Asthma: 1 patients, 6 cycles


,diagnosis,patient_id,cycles,recordings,chest_locations
0,Asthma,103,6,1,1



Bronchiectasis: 6 patients, 90 cycles


,diagnosis,patient_id,cycles,recordings,chest_locations
0,Bronchiectasis,201,41,3,2
1,Bronchiectasis,111,17,2,1
2,Bronchiectasis,215,11,2,2
3,Bronchiectasis,116,8,1,2
4,Bronchiectasis,168,7,1,1
5,Bronchiectasis,196,6,1,1



Bronchiolitis: 3 patients, 74 cycles


...

,diagnosis,patient_id,cycles,recordings,chest_locations
0,Pneumonia,135,112,3,5
1,Pneumonia,122,52,3,3
2,Pneumonia,140,40,2,2
3,Pneumonia,226,31,1,3
4,Pneumonia,219,26,3,2
5,Pneumonia,191,24,2,3



URTI: 10 patients, 171 cycles


,diagnosis,patient_id,cycles,recordings,chest_locations
0,URTI,188,62,1,4
1,URTI,210,31,1,2
2,URTI,137,17,1,2
3,URTI,131,11,1,1
4,URTI,148,11,1,1
5,URTI,164,10,1,1
6,URTI,129,9,1,1
7,URTI,105,8,1,1
8,URTI,197,8,1,2
9,URTI,190,4,1,1



Overall patient-level class support:


,patients,total_cycles,min_cycles_per_patient,median_cycles_per_patient,max_cycles_per_patient
diagnosis,,,,,
Asthma,1,6,6,6.0,6
Bronchiectasis,6,90,6,9.5,41
Bronchiolitis,3,74,19,27.0,28
COPD,37,3220,5,57.0,507
Healthy,12,147,4,9.5,42
LRTI,2,32,8,16.0,24
Pneumonia,6,285,24,35.5,112
URTI,10,171,4,10.5,62


In [13]:
# Check whether Configuration 4 OOF predictions are available and aligned.
# Do not calculate recording-level metrics until this check passes.

print("Development rows:", len(development_features))

if "oof_pred_config4" not in globals():
    print(
        "\nSTOP: oof_pred_config4 was not found. "
        "We need to recover or regenerate Configuration 4 OOF predictions."
    )
else:
    print("OOF prediction rows:", len(oof_pred_config4))

    oof_array = np.asarray(oof_pred_config4)

    print("OOF prediction shape:", oof_array.shape)

    if len(oof_array) != len(development_features):
        print(
            "\nSTOP: OOF predictions and development rows have different "
            "lengths. Do not aggregate them."
        )
    elif oof_array.ndim != 1:
        print(
            "\nThe stored predictions are not a one-dimensional class-label "
            "array. We need to inspect their format before aggregation."
        )
    else:
        print("\nPrediction alignment by row count passes.")
        print("Predicted class IDs:", np.unique(oof_array))
        print("Expected class IDs:", list(range(len(ALL_CLASSES))))

Development rows: 4025

STOP: oof_pred_config4 was not found. We need to recover or regenerate Configuration 4 OOF predictions.


In [14]:
# Check the objects needed to regenerate Configuration 4 OOF predictions

required_objects = [
    "X_dev",
    "y_encoded",
    "groups_dev",
    "fold_splits",
    "ALL_CLASSES",
    "class_to_id",
    "id_to_class",
]

for name in required_objects:
    exists = name in globals()
    print(f"{name}: {'Available' if exists else 'MISSING'}")

print("\nXGBoost version:", xgb.__version__ if "xgb" in globals() else "xgboost import missing")

if "fold_splits" in globals():
    print("Number of existing folds:", len(fold_splits))

if "X_dev" in globals():
    print("Feature matrix shape:", X_dev.shape)

if "y_encoded" in globals():
    print("Encoded labels shape:", y_encoded.shape)

if "groups_dev" in globals():
    print("Unique patients:", groups_dev.nunique())

X_dev: Available
y_encoded: Available
groups_dev: Available
fold_splits: Available
ALL_CLASSES: Available
class_to_id: Available
id_to_class: Available

XGBoost version: 3.4.1
Number of existing folds: 3
Feature matrix shape: (4025, 39)
Encoded labels shape: (4025,)
Unique patients: 77


In [15]:
# Regenerate leakage-safe OOF predictions for Configuration 4

import numpy as np
import xgboost as xgb

from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Configuration 4: selected from the previous grouped-CV search
CONFIG4 = {
    "n_estimators": 400,
    "learning_rate": 0.03,
    "max_depth": 3,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.5,
    "reg_lambda": 10.0,
}

n_samples = len(X_dev)
n_classes = len(ALL_CLASSES)

oof_pred_config4 = np.full(n_samples, -1, dtype=int)
oof_proba_config4 = np.zeros((n_samples, n_classes), dtype=float)

for fold_no, (train_idx, val_idx) in enumerate(fold_splits, start=1):
    X_train = X_dev.iloc[train_idx]
    X_val = X_dev.iloc[val_idx]

    y_train_global = y_encoded.iloc[train_idx].to_numpy()
    y_val_global = y_encoded.iloc[val_idx].to_numpy()

    # Only classes present in this training fold
    present_classes = np.sort(np.unique(y_train_global))
    global_to_local = {
        int(global_id): local_id
        for local_id, global_id in enumerate(present_classes)
    }

    y_train_local = np.array(
        [global_to_local[int(label)] for label in y_train_global],
        dtype=int
    )

    # Calculate class-balanced weights using training data only
    sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train_global
    )

    model = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=len(present_classes),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        **CONFIG4
    )

    model.fit(
        X_train,
        y_train_local,
        sample_weight=sample_weights,
        verbose=False
    )

    local_proba = model.predict_proba(X_val)
    local_pred = np.argmax(local_proba, axis=1)

    # Convert predictions back to the global class IDs
    global_pred = present_classes[local_pred]
    oof_pred_config4[val_idx] = global_pred

    # Preserve global class-column order; absent classes get probability 0
    for local_id, global_id in enumerate(present_classes):
        oof_proba_config4[val_idx, int(global_id)] = local_proba[:, local_id]

    print(
        f"Fold {fold_no}: "
        f"train patients={groups_dev.iloc[train_idx].nunique()}, "
        f"validation patients={groups_dev.iloc[val_idx].nunique()}, "
        f"validation rows={len(val_idx)}, "
        f"training classes={len(present_classes)}"
    )

assert np.all(oof_pred_config4 >= 0), "Some rows lack OOF predictions."
assert np.allclose(oof_proba_config4.sum(axis=1), 1.0, atol=1e-5)

y_true_global = y_encoded.to_numpy()

print("\n=== Configuration 4: regenerated pooled OOF results ===")
print(f"Accuracy:          {accuracy_score(y_true_global, oof_pred_config4):.4f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_true_global, oof_pred_config4):.4f}")
print(f"Macro-F1:           {f1_score(y_true_global, oof_pred_config4, labels=np.arange(n_classes), average='macro', zero_division=0):.4f}")
print(f"Weighted F1:        {f1_score(y_true_global, oof_pred_config4, labels=np.arange(n_classes), average='weighted', zero_division=0):.4f}")

print("\n=== Classification report ===")
print(
    classification_report(
        y_true_global,
        oof_pred_config4,
        labels=np.arange(n_classes),
        target_names=ALL_CLASSES,
        zero_division=0,
        digits=4
    )
)

print("OOF prediction shape:", oof_pred_config4.shape)
print("OOF probability shape:", oof_proba_config4.shape)

Fold 1: train patients=55, validation patients=22, validation rows=1340, training classes=8
Fold 2: train patients=50, validation patients=27, validation rows=1346, training classes=8
Fold 3: train patients=49, validation patients=28, validation rows=1339, training classes=7

=== Configuration 4: regenerated pooled OOF results ===
Accuracy:          0.7478
Balanced accuracy: 0.2756
Macro-F1:           0.2518
Weighted F1:        0.7644

=== Classification report ===
                precision    recall  f1-score   support

        Asthma     0.0000    0.0000    0.0000         6
Bronchiectasis     0.2323    0.2556    0.2434        90
 Bronchiolitis     0.0000    0.0000    0.0000        74
          COPD     0.9296    0.8618    0.8944      3220
       Healthy     0.2966    0.5306    0.3805       147
          LRTI     0.0000    0.0000    0.0000        32
     Pneumonia     0.2622    0.3404    0.2962       285
          URTI     0.1859    0.2164    0.2000       171

      accuracy          

In [16]:
# Compare cycle-level predictions with recording-level probability aggregation

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

# Attach OOF probabilities and predictions to the corresponding feature rows
oof_results = development_features[
    ["patient_id", "recording_id", "diagnosis"]
].copy().reset_index(drop=True)

oof_results["true_class_id"] = y_encoded.reset_index(drop=True).to_numpy()
oof_results["predicted_class_id"] = oof_pred_config4

for class_id, class_name in enumerate(ALL_CLASSES):
    oof_results[f"prob_{class_id}"] = oof_proba_config4[:, class_id]

probability_columns = [
    f"prob_{class_id}" for class_id in range(len(ALL_CLASSES))
]

# Cycle-level metrics
y_true_cycle = oof_results["true_class_id"].to_numpy()
y_pred_cycle = oof_results["predicted_class_id"].to_numpy()

cycle_metrics = {
    "level": "Cycle",
    "units": len(oof_results),
    "accuracy": accuracy_score(y_true_cycle, y_pred_cycle),
    "balanced_accuracy": balanced_accuracy_score(y_true_cycle, y_pred_cycle),
    "macro_f1": f1_score(
        y_true_cycle, y_pred_cycle,
        labels=np.arange(len(ALL_CLASSES)),
        average="macro",
        zero_division=0
    ),
    "weighted_f1": f1_score(
        y_true_cycle, y_pred_cycle,
        labels=np.arange(len(ALL_CLASSES)),
        average="weighted",
        zero_division=0
    )
}

# Aggregate probabilities within each patient-recording-diagnosis group
recording_results = (
    oof_results
    .groupby(
        ["patient_id", "recording_id", "diagnosis"],
        as_index=False
    )[probability_columns]
    .mean()
)

recording_results["true_class_id"] = (
    recording_results["diagnosis"].map(class_to_id).astype(int)
)

recording_results["predicted_class_id"] = (
    recording_results[probability_columns]
    .to_numpy()
    .argmax(axis=1)
)

y_true_recording = recording_results["true_class_id"].to_numpy()
y_pred_recording = recording_results["predicted_class_id"].to_numpy()

recording_metrics = {
    "level": "Recording",
    "units": len(recording_results),
    "accuracy": accuracy_score(y_true_recording, y_pred_recording),
    "balanced_accuracy": balanced_accuracy_score(
        y_true_recording, y_pred_recording
    ),
    "macro_f1": f1_score(
        y_true_recording, y_pred_recording,
        labels=np.arange(len(ALL_CLASSES)),
        average="macro",
        zero_division=0
    ),
    "weighted_f1": f1_score(
        y_true_recording, y_pred_recording,
        labels=np.arange(len(ALL_CLASSES)),
        average="weighted",
        zero_division=0
    )
}

comparison = pd.DataFrame([cycle_metrics, recording_metrics])
display(comparison.round(4))

print("\n=== Recording-level classification report ===")
print(
    classification_report(
        y_true_recording,
        y_pred_recording,
        labels=np.arange(len(ALL_CLASSES)),
        target_names=ALL_CLASSES,
        zero_division=0,
        digits=4
    )
)

print("\nCycle rows:", len(oof_results))
print("Patient-recording groups:", len(recording_results))
print("Unique patients:", recording_results["patient_id"].nunique())

,level,units,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,Cycle,4025,0.7478,0.2756,0.2518,0.7644
1,Recording,154,0.6818,0.2895,0.2809,0.6711



=== Recording-level classification report ===
                precision    recall  f1-score   support

        Asthma     0.0000    0.0000    0.0000         1
Bronchiectasis     1.0000    0.2000    0.3333        10
 Bronchiolitis     0.0000    0.0000    0.0000         3
          COPD     0.8529    0.8614    0.8571       101
       Healthy     0.5263    0.7692    0.6250        13
          LRTI     0.0000    0.0000    0.0000         2
     Pneumonia     0.2222    0.2857    0.2500        14
          URTI     0.1667    0.2000    0.1818        10

      accuracy                         0.6818       154
     macro avg     0.3460    0.2895    0.2809       154
  weighted avg     0.6998    0.6818    0.6711       154


Cycle rows: 4025
Patient-recording groups: 154
Unique patients: 77


In [17]:
# Patient-level aggregation of out-of-fold probabilities

patient_results = (
    oof_results
    .groupby(["patient_id", "diagnosis"], as_index=False)[probability_columns]
    .mean()
)

patient_results["true_class_id"] = (
    patient_results["diagnosis"].map(class_to_id).astype(int)
)

patient_results["predicted_class_id"] = (
    patient_results[probability_columns]
    .to_numpy()
    .argmax(axis=1)
)

y_true_patient = patient_results["true_class_id"].to_numpy()
y_pred_patient = patient_results["predicted_class_id"].to_numpy()

patient_metrics = {
    "level": "Patient",
    "units": len(patient_results),
    "accuracy": accuracy_score(y_true_patient, y_pred_patient),
    "balanced_accuracy": balanced_accuracy_score(
        y_true_patient, y_pred_patient
    ),
    "macro_f1": f1_score(
        y_true_patient, y_pred_patient,
        labels=np.arange(len(ALL_CLASSES)),
        average="macro",
        zero_division=0
    ),
    "weighted_f1": f1_score(
        y_true_patient, y_pred_patient,
        labels=np.arange(len(ALL_CLASSES)),
        average="weighted",
        zero_division=0
    )
}

display(pd.DataFrame([cycle_metrics, recording_metrics, patient_metrics]).round(4))

print("\n=== Patient-level classification report ===")
print(
    classification_report(
        y_true_patient,
        y_pred_patient,
        labels=np.arange(len(ALL_CLASSES)),
        target_names=ALL_CLASSES,
        zero_division=0,
        digits=4
    )
)

print("\nPatient count:", len(patient_results))
print("Unique patients:", patient_results["patient_id"].nunique())

,level,units,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,Cycle,4025,0.7478,0.2756,0.2518,0.7644
1,Recording,154,0.6818,0.2895,0.2809,0.6711
2,Patient,77,0.5844,0.2860,0.2741,0.5538



=== Patient-level classification report ===
                precision    recall  f1-score   support

        Asthma     0.0000    0.0000    0.0000         1
Bronchiectasis     1.0000    0.1667    0.2857         6
 Bronchiolitis     0.0000    0.0000    0.0000         3
          COPD     0.7561    0.8378    0.7949        37
       Healthy     0.6000    0.7500    0.6667        12
          LRTI     0.0000    0.0000    0.0000         2
     Pneumonia     0.1818    0.3333    0.2353         6
          URTI     0.2222    0.2000    0.2105        10

      accuracy                         0.5844        77
     macro avg     0.3450    0.2860    0.2741        77
  weighted avg     0.5778    0.5844    0.5538        77


Patient count: 77
Unique patients: 77
